# 🗂️ EnerGIS Workflow Browser

Interaktiver Browser für gespeicherte Simulationen.

## 🎯 Features

- **📁 Automatisches Scannen** aller Simulationen in `saved_workflows/`
- **🔍 Dropdown-Auswahl** zur Simulation-Auswahl
- **📊 Vollständiges Dashboard** für jede Simulation (alle Tabs)
- **📈 Zugriff auf CSV-Daten** und exportierte Plots
- **🔀 Vergleichs-Modus** (coming soon)

## 🚀 Quick Start

1. Führe die Setup-Zelle aus
2. Wähle eine Simulation im Dropdown
3. Analysiere die Ergebnisse in den Tabs

---

## 📦 Setup

In [ ]:
# Minimal-Bootstrap: Füge Projekt-Root zu sys.path hinzu
import sys
from pathlib import Path

# Finde Projekt-Root
current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'energis').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        PROJECT_ROOT = candidate
        break

print(f"✅ Projekt-Root: {PROJECT_ROOT}")

In [ ]:
# Imports
from energis.io.workflow_browser import create_workflow_browser, HAVE_PANEL, HAVE_PLOTLY

print(f"✅ Imports erfolgreich")
print(f"   Panel: {HAVE_PANEL}")
print(f"   Plotly: {HAVE_PLOTLY}")

if not HAVE_PANEL:
    print("\n❌ Panel nicht installiert!")
    print("   Installation: pip install panel holoviews bokeh plotly")
    raise ImportError("Panel required")

## 🗂️ Workflow Browser erstellen

In [ ]:
# Browser erstellen
browser = create_workflow_browser(
    saved_workflows_dir="saved_workflows",
    title="EnerGIS Workflow Browser - Gespeicherte Simulationen"
)

print("✅ Browser erstellt!")
print("\n💡 Verwendung:")
print("   • Evaluiere die nächste Zelle, um den Browser anzuzeigen")
print("   • Oder starte als Webapp: panel serve workflow_browser.ipynb --show")

## 🎛️ Browser anzeigen

**Hinweis:** In VS-Code funktioniert Panel am besten im Browser!

**Empfohlen:** Starte mit `panel serve workflow_browser.ipynb --show` im Terminal

In [ ]:
# Browser anzeigen
browser

---

## 📚 Erweiterte Verwendung

### Workflows manuell laden und analysieren

Falls du direkten Zugriff auf die Daten brauchst:

In [ ]:
# Optional: Manueller Zugriff auf gespeicherte Workflows
from energis.io.notebook_helpers import list_saved_workflows, load_workflow_from_saved

# Liste alle Workflows
workflows = list_saved_workflows(sort_by="date")

print(f"📦 {len(workflows)} Workflows gefunden:\n")
print(f"{'#':<4} {'Name':<50} {'Kosten [EUR]'}")
print("-" * 80)

for i, wf in enumerate(workflows[:10], 1):  # Zeige erste 10
    name = wf['name'][:48] + ".." if len(wf['name']) > 50 else wf['name']
    costs = f"{wf['costs']:,.0f}" if wf['costs'] > 0 else 'N/A'
    print(f"{i:<4} {name:<50} {costs}")

In [ ]:
# Lade einen spezifischen Workflow (ändere Index nach Bedarf)
WORKFLOW_INDEX = 0  # Erster Workflow in der Liste

if workflows:
    selected_wf = workflows[WORKFLOW_INDEX]
    
    print(f"📂 Lade: {selected_wf['name']}")
    
    workflow = load_workflow_from_saved(selected_wf['path'])
    
    print(f"✅ Workflow geladen!")
    print(f"   Steps: {' → '.join(workflow.plan.steps)}")
    print(f"   Has PF: {workflow.pf_result is not None}")
    print(f"   Has RH: {workflow.rh_result is not None}")
    
    # Jetzt kannst du mit 'workflow' arbeiten
    # z.B. Dashboard erstellen:
    # from energis.io.dashboard import create_dashboard
    # dashboard = create_dashboard(workflow)
    # dashboard
else:
    print("⚠️ Keine Workflows gefunden")

### CSV-Daten direkt laden

Jede Simulation hat exportierte CSV-Dateien:

In [ ]:
# CSV-Daten laden (unabhängig vom workflow.pkl)
import pandas as pd

if workflows:
    workflow_path = workflows[WORKFLOW_INDEX]['path']
    
    # RH Zeitreihen
    rh_csv = workflow_path / 'rh_timeseries.csv'
    
    if rh_csv.exists():
        df = pd.read_csv(rh_csv, index_col=0, parse_dates=True)
        
        print(f"✅ CSV geladen: {rh_csv.name}")
        print(f"   Zeilen: {len(df):,}")
        print(f"   Spalten: {len(df.columns)}")
        print(f"\n   Erste Spalten: {', '.join(df.columns[:10])}")
        
        # Zeige erste Zeilen
        display(df.head())
    else:
        print(f"⚠️ CSV nicht gefunden: {rh_csv}")

---

## 💡 Tipps

**Browser-Modus (empfohlen):**
```bash
# Im Terminal
panel serve notebooks/workflow_browser.ipynb --show
```

**Verfügbare Dateien pro Simulation:**
- `workflow.pkl` - Komplettes Workflow-Objekt
- `metadata.json` - Metadaten (Name, Datum, Kosten, etc.)
- `pf_timeseries.csv` - Perfect Forecast Zeitreihen
- `rh_timeseries.csv` - Rolling Horizon Zeitreihen
- `design.json` - Anlagen-Design
- `*.pdf`, `*.svg` - Exportierte Plots

**Neue Simulationen erstellen:**
- `notebooks/scenario_studio.ipynb` - Interaktive Szenario-Analyse
- `notebooks/runner.ipynb` - Batch-Runs

---